# 🚀 Huấn luyện EDR-REDNet trên Kaggle (Bản sửa lỗi 100%)
Notebook này đã được Fix toàn bộ các lỗi liên quan đến Kaggle (Đường dẫn, GPU, Thư viện, Config).

**Lưu ý trước khi chạy:**
1. Đảm bảo anh đã đẩy code mới nhất lên Github.
2. Bật GPU P100 ở cột bên phải (Session options -> Accelerator).
3. Đã Add thư mục Data `AAPM-Mayo Clinic` vào Kaggle.

In [ ]:
# 1. Cài đặt các thư viện phụ trợ
!pip install wandb pydicom pyyaml
# Đã xóa lệnh cài PyTorch để dùng bản CUDA gốc của Kaggle.

In [ ]:
import os

# 2. Tải Code từ GitHub về Kaggle
GIT_REPO_URL = "https://github.com/minhvuongle2004/ldct-benchmark.git"
WORKING_DIR = "/kaggle/working/ldct-benchmark"

if not os.path.exists(WORKING_DIR):
    print("Đang clone source code từ GitHub...")
    !git clone {GIT_REPO_URL}
    print("Clone hoàn tất!")
else:
    print("Code đã tồn tại, đang pull cập nhật mới nhất...")
    os.chdir(WORKING_DIR)
    !git pull

os.chdir(WORKING_DIR)
print("Thư mục hiện tại:", os.getcwd())

In [ ]:
# 3. Tự động quét đường dẫn Data siêu chuẩn
import os
import glob

# Tự động tìm thư mục LDCT-and-Projection-data ở bất cứ đâu trong /kaggle/input/
found_paths = glob.glob('/kaggle/input/**/LDCT-and-Projection-data', recursive=True)

if len(found_paths) == 0:
    print("❌ KHÔNG TÌM THẤY DỮ LIỆU! Anh hãy kiểm tra lại xem đã Add Data vào notebook chưa nhé!")
else:
    # Lấy thư mục cha của LDCT-and-Projection-data
    DATA_DATASET_PATH = found_paths[0].replace('/LDCT-and-Projection-data', '')
    os.environ['LDCTBENCH_DATAFOLDER'] = DATA_DATASET_PATH
    print("✅ Đã tìm thấy và Set đường dẫn Data thành công:")
    print(os.environ['LDCTBENCH_DATAFOLDER'])

In [ ]:
# 4. Sửa lỗi Config phản chủ
# Xóa bỏ dòng `datafolder: data` trong file yaml để nó chịu nhận đường dẫn Kaggle
!sed -i 's/datafolder: data/datafolder: ""/g' /kaggle/working/ldct-benchmark/configs/edrrednet.yaml
print("Đã Fix lỗi Config!")

In [ ]:
# 5. BẮT ĐẦU TRAIN EDR-REDNet VỚI NHIỀU SEED 🚀
%cd /kaggle/working/ldct-benchmark
# Chạy vòng lặp Train cho 2 seed: 42 và 2024 liên tiếp
!for SEED in 42 2024; do \
    echo "========================================="; \
    echo "🚀 BẮT ĐẦU TRAIN VỚI SEED: $SEED"; \
    echo "========================================="; \
    sed -i "s/seed: .*/seed: $SEED/g" configs/edrrednet.yaml; \
    python -m ldctbench.scripts.train --config configs/edrrednet.yaml --dryrun; \
    cp wandb/*/files/best_SSIM.pt /kaggle/working/seed${SEED}_best_SSIM.pt; \
    rm -rf wandb/*; \
done


In [ ]:
# 6. KIỂM TRA FILE TRỌNG SỐ (.pt)
# File trọng số của các Seed sẽ được đổi tên và cất an toàn ở thư mục gốc
!ls -lh /kaggle/working/*.pt